# 04 — Cross-Product Comparison & Capital Integration

This notebook brings all three products together:

| Section | Description |
|---------|-------------|
| 1 | Generate all datasets and run full LGD pipeline |
| 2 | Cross-product LGD comparison |
| 3 | Portfolio-level summary |
| 4 | Regulatory overlay impact analysis |
| 5 | Illustrative expected loss and capital |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_all_datasets
from src.lgd_calculation import (
    MortgageLGDEngine, CommercialLGDEngine, DevelopmentLGDEngine,
    run_full_pipeline, exposure_weighted_average
)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

## 1. Generate All Data & Run Pipeline

In [ ]:
datasets = generate_all_datasets()
results = run_full_pipeline(datasets)

for product, data in datasets.items():
    n_loans = len(data['loans'])
    n_cfs = len(data['cashflows'])
    total_ead = data['loans']['ead'].sum()
    print(f"{product:15s}: {n_loans:>4} loans, {n_cfs:>6} cashflows, EAD=${total_ead:>15,.0f}")

## 2. Cross-Product LGD Comparison

In [ ]:
# Combine all products
comparison = []
for product, res in results.items():
    df = res['loans_with_overlays']
    comparison.append({
        'Product': product.title(),
        'N Loans': len(df),
        'Total EAD': df['ead'].sum(),
        'Realised LGD (EWA)': exposure_weighted_average(df, 'realised_lgd'),
        'Downturn LGD (EWA)': exposure_weighted_average(df, 'lgd_downturn'),
        'Final LGD (EWA)': exposure_weighted_average(df, 'lgd_final'),
        'Mean Workout (months)': df['workout_months'].mean(),
    })

comparison_df = pd.DataFrame(comparison)
print("=== Cross-Product Comparison ===")
display(comparison_df.round(4))

In [ ]:
# LGD distributions side-by-side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colours = {'mortgage': '#3498db', 'commercial': '#e67e22', 'development': '#e74c3c'}

for idx, (product, res) in enumerate(results.items()):
    df = res['loans_with_overlays']
    axes[idx].hist(df['realised_lgd'], bins=35, edgecolor='white', alpha=0.8, color=colours[product])
    mean_lgd = df['realised_lgd'].mean()
    axes[idx].axvline(mean_lgd, color='red', ls='--', label=f'Mean: {mean_lgd:.2%}')
    axes[idx].set_title(f'{product.title()} — Realised LGD')
    axes[idx].set_xlabel('LGD')
    axes[idx].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'cross_product_lgd_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall: Realised → Final across products
fig, ax = plt.subplots(figsize=(12, 6))

products = ['Mortgage', 'Commercial', 'Development']
x = np.arange(len(products))
width = 0.25

realised = [comparison_df.loc[comparison_df['Product'] == p, 'Realised LGD (EWA)'].values[0] for p in products]
downturn = [comparison_df.loc[comparison_df['Product'] == p, 'Downturn LGD (EWA)'].values[0] for p in products]
final = [comparison_df.loc[comparison_df['Product'] == p, 'Final LGD (EWA)'].values[0] for p in products]

ax.bar(x - width, realised, width, label='Realised LGD', color='#3498db')
ax.bar(x, downturn, width, label='Downturn LGD', color='#e67e22')
ax.bar(x + width, final, width, label='Final LGD', color='#e74c3c')

ax.set_xticks(x)
ax.set_xticklabels(products)
ax.set_ylabel('Exposure-Weighted LGD')
ax.set_title('Cross-Product LGD: Realised → Downturn → Final')
ax.legend()

# Add value labels
for bars_group in [realised, downturn, final]:
    for i, v in enumerate(bars_group):
        offset = [-width, 0, width][[realised, downturn, final].index(bars_group)]
        ax.text(i + offset, v + 0.005, f'{v:.1%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'cross_product_waterfall.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Portfolio-Level Summary

In [ ]:
# Combined portfolio
all_loans = pd.concat([
    results['mortgage']['loans_with_overlays'][['product', 'ead', 'realised_lgd', 'lgd_final', 'workout_months']],
    results['commercial']['loans_with_overlays'][['product', 'ead', 'realised_lgd', 'lgd_final', 'workout_months']],
    results['development']['loans_with_overlays'][['product', 'ead', 'realised_lgd', 'lgd_final', 'workout_months']],
], ignore_index=True)

portfolio_ewa = exposure_weighted_average(all_loans, 'lgd_final')
portfolio_ead = all_loans['ead'].sum()

print(f"=== Portfolio Summary ===")
print(f"Total loans:            {len(all_loans):,}")
print(f"Total EAD:              ${portfolio_ead:,.0f}")
print(f"Portfolio EWA Final LGD: {portfolio_ewa:.2%}")
print(f"\n--- By Product ---")
for product in ['Mortgage', 'Commercial', 'Development']:
    sub = all_loans[all_loans['product'] == product]
    ewa = exposure_weighted_average(sub, 'lgd_final')
    pct_ead = sub['ead'].sum() / portfolio_ead
    print(f"  {product:15s}: EAD={sub['ead'].sum():>15,.0f} ({pct_ead:.1%})  Final LGD={ewa:.2%}  n={len(sub)}")

## 4. Regulatory Overlay Impact

Quantify the impact of each regulatory overlay step (downturn, MoC, product-specific).

In [ ]:
impact = []
for product, res in results.items():
    df = res['loans_with_overlays']
    r = exposure_weighted_average(df, 'realised_lgd')
    d = exposure_weighted_average(df, 'lgd_downturn')
    f = exposure_weighted_average(df, 'lgd_final')
    impact.append({
        'Product': product.title(),
        'Realised LGD': r,
        'Downturn Impact': d - r,
        'MoC + Overlay Impact': f - d,
        'Total Overlay': f - r,
        'Final LGD': f,
        'Overlay %': (f - r) / r if r > 0 else 0,
    })

impact_df = pd.DataFrame(impact)
print("=== Regulatory Overlay Impact ===")
display(impact_df.round(4))

## 4b. Industry Risk Impact Across Products

The industry risk integration affects Commercial and Development products differently.
This section quantifies the cross-product impact of industry-sensitive overlays on LGD
and identifies which industries drive the largest adjustments.

In [ ]:
# Cross-product industry risk impact analysis
from src.industry_risk_integration import (
    IndustryRiskLoader, enrich_loans_with_industry_risk,
    compute_industry_downturn_scalar, compute_industry_recovery_haircut,
)

INDUSTRY_DATA_PATH = os.path.join(os.getcwd(), "..", "data", "external", "industry_risk")
loader = IndustryRiskLoader(INDUSTRY_DATA_PATH)

# Enrich commercial and development loans
com_loans = datasets["commercial"]["loans"]
dev_loans = datasets["development"]["loans"]
com_enriched = enrich_loans_with_industry_risk(com_loans, loader, "commercial")
dev_enriched = enrich_loans_with_industry_risk(dev_loans, loader, "development")

# Run enhanced pipeline
com_engine = CommercialLGDEngine()
dev_engine = DevelopmentLGDEngine()
com_enhanced = com_engine.apply_overlays(com_enriched)
dev_enhanced = dev_engine.apply_overlays(dev_enriched)

# Baseline (already computed in results)
com_baseline = results["commercial"]["loans_with_overlays"]
dev_baseline = results["development"]["loans_with_overlays"]

# Industry risk impact summary
print("=== Industry Risk Impact on LGD ===")
for name, base, enh in [("Commercial", com_baseline, com_enhanced),
                         ("Development", dev_baseline, dev_enhanced)]:
    base_lgd = exposure_weighted_average(base, "lgd_final")
    enh_lgd = exposure_weighted_average(enh, "lgd_final")
    diff_pp = (enh_lgd - base_lgd) * 100
    print(f"  {name:15s}: Baseline={base_lgd:.2%}  Enhanced={enh_lgd:.2%}  Delta={diff_pp:+.1f}pp")

# Which industries drive the largest LGD adjustments?
print("
=== Largest Industry LGD Adjustments (Commercial) ===")
com_by_ind = com_enriched.copy()
com_by_ind["lgd_baseline"] = com_baseline["lgd_final"].values
com_by_ind["lgd_enhanced"] = com_enhanced["lgd_final"].values
com_by_ind["lgd_delta_pp"] = (com_by_ind["lgd_enhanced"] - com_by_ind["lgd_baseline"]) * 100

ind_impact = com_by_ind.groupby("industry").agg(
    risk_score=("industry_risk_score", "first"),
    mean_delta_pp=("lgd_delta_pp", "mean"),
    total_ead=("ead", "sum"),
    count=("loan_id", "count"),
).sort_values("mean_delta_pp", ascending=False).round(2)
display(ind_impact)


## 5. Illustrative Expected Loss & Capital

In [ ]:
# Illustrative PD assumptions by product
pd_assumptions = {'Mortgage': 0.015, 'Commercial': 0.030, 'Development': 0.050}

el_summary = []
for product, res in results.items():
    df = res['loans_with_overlays']
    pd_est = pd_assumptions[product.title()]
    lgd_ewa = exposure_weighted_average(df, 'lgd_final')
    total_ead = df['ead'].sum()
    el = pd_est * lgd_ewa * total_ead
    
    el_summary.append({
        'Product': product.title(),
        'PD': pd_est,
        'LGD (EWA)': lgd_ewa,
        'Total EAD': total_ead,
        'Expected Loss': el,
        'EL Rate (bps)': pd_est * lgd_ewa * 10000,
    })

el_df = pd.DataFrame(el_summary)
print("=== Illustrative Expected Loss ===")
print("EL = PD × LGD × EAD")
display(el_df.round(4))

total_el = el_df['Expected Loss'].sum()
total_ead_all = el_df['Total EAD'].sum()
print(f"\nPortfolio Expected Loss: ${total_el:,.0f}")
print(f"Portfolio EL Rate: {total_el / total_ead_all * 10000:.1f} bps")

In [ ]:
# EAD and EL composition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colours = ['#3498db', '#e67e22', '#e74c3c']

# EAD pie
axes[0].pie(el_df['Total EAD'], labels=el_df['Product'], autopct='%1.1f%%',
            colors=colours, startangle=90)
axes[0].set_title('EAD Composition')

# EL pie
axes[1].pie(el_df['Expected Loss'], labels=el_df['Product'], autopct='%1.1f%%',
            colors=colours, startangle=90)
axes[1].set_title('Expected Loss Composition')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'portfolio_ead_el_composition.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save portfolio summary
comparison_df.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'cross_product_comparison.csv'), index=False)
el_df.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'portfolio_expected_loss.csv'), index=False)
impact_df.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'regulatory_overlay_impact.csv'), index=False)

print("All portfolio outputs saved to outputs/tables/")